In [ ]:
import gradio as gr
from PIL import Image
import os
import torch
import pandas as pd
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms
import gradio as gr
import json
import torch.nn as nn


C:\Users\USER\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json
import os

# Path to your vocabulary JSON file
vocab_json_path = 'vocab.json'

# Check if the file exists before loading
with open(vocab_json_path, 'r') as f:
    vocab_dict = json.load(f)

    num2char = vocab_dict['idx2char']
    char2num = vocab_dict['char2idx']

    print("Vocabulary loaded from vocab.json")
    print("num2char:", num2char)
    print("char2num:", char2num)



Vocabulary loaded from vocab.json
num2char: {'0': '!', '1': '$', '2': '%', '3': '&', '4': "'", '5': '(', '6': ')', '7': '+', '8': ',', '9': '-', '10': '.', '11': '/', '12': '0', '13': '1', '14': '2', '15': '3', '16': '4', '17': '5', '18': '6', '19': '7', '20': '8', '21': '9', '22': ':', '23': '?', '24': 'A', '25': 'B', '26': 'C', '27': 'D', '28': 'E', '29': 'F', '30': 'G', '31': 'H', '32': 'I', '33': 'J', '34': 'K', '35': 'L', '36': 'M', '37': 'N', '38': 'O', '39': 'P', '40': 'Q', '41': 'R', '42': 'S', '43': 'T', '44': 'U', '45': 'V', '46': 'W', '47': 'X', '48': 'Y', '49': 'Z', '50': '[', '51': ']', '52': 'a', '53': 'b', '54': 'c', '55': 'd', '56': 'e', '57': 'f', '58': 'g', '59': 'h', '60': 'i', '61': 'j', '62': 'k', '63': 'l', '64': 'm', '65': 'n', '66': 'o', '67': 'p', '68': 'q', '69': 'r', '70': 's', '71': 't', '72': 'u', '73': 'v', '74': 'w', '75': 'x', '76': 'y', '77': 'z', '78': '«', '79': '»', '80': 'ç', '81': 'è', '82': 'é', '83': 'ក', '84': 'ខ', '85': 'គ', '86': 'ឃ', '87': 'ង

In [3]:
# Define the transformation operations
transform_ops = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

# Function to remove duplicates in the predicted text
def remove_duplicates(text):
    if len(text) > 1:
        letters = [text[0]] + [letter for idx, letter in enumerate(text[1:], start=1) if text[idx] != text[idx-1]]
    elif len(text) == 1:
        letters = [text[0]]
    else:
        return ""
    return "".join(letters)

# Function to correct the predicted word
def correct_prediction(word):
    parts = word.split("-")
    parts = [remove_duplicates(part) for part in parts]
    corrected_word = "".join(parts)
    return corrected_word

# Function to decode predictions from logits
def decode_predictions(text_batch_logits, num2char):
    text_batch_tokens = F.softmax(text_batch_logits, 2).argmax(2)  # [T, batch_size]
    text_batch_tokens = text_batch_tokens.numpy().T  # [batch_size, T]

    text_batch_tokens_new = []
    for text_tokens in text_batch_tokens:
        text = [num2char[str(idx)] for idx in text_tokens]  # Ensure num2char is defined
        text = "".join(text)
        text_batch_tokens_new.append(text)

    return text_batch_tokens_new

In [4]:
# Define the OCR model class
class OCR_CNN_GRU(nn.Module):
    def __init__(self, num_chars, rnn_hidden_size=512, dropout=0.1):
        super(OCR_CNN_GRU, self).__init__()
        self.num_chars = num_chars
        self.rnn_hidden_size = rnn_hidden_size
        self.dropout = dropout
        
        # CNN layers
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=2),
            
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=2),
        )

        self.linear1 = nn.Linear(3200, rnn_hidden_size)
        
        # GRU layers
        self.gru = nn.GRU(
            input_size=rnn_hidden_size, 
            hidden_size=rnn_hidden_size, 
            num_layers=2, 
            bidirectional=True, 
            batch_first=True
        )
        
        # Output layer
        self.linear2 = nn.Linear(rnn_hidden_size * 2, num_chars)
        
    def forward(self, x):
        x = self.cnn(x)
        
        # Reshape from [batch_size, channels, height, width] to [batch_size, width, channels * height]
        batch_size, channels, height, width = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, width, -1)
        
        # Linear layer to reduce dimension
        x = self.linear1(x)
        
        # GRU layers
        x, _ = self.gru(x)
        
        # Output layer
        x = self.linear2(x)
        
        # Transpose to get the shape [sequence_length, batch_size, num_classes]
        x = x.transpose(0, 1)
        
        return x

# Initialize and load the model
num_chars = len(num2char)  # Replace idx2char with the correct length of your character set
model = OCR_CNN_GRU(num_chars=num_chars)
model.load_state_dict(torch.load('checkpoint.pth', map_location=torch.device('cpu')))
model.eval()


OCR_CNN_GRU(
  (cnn): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (linear1): Linear(in_features=3200, out_features=512, bias=True)
  (gru): GRU(512, 512, num_layers=2, batch_first=True, bidirectional=True)
  (linear2): Linear(in_features=1024, out_features=172, bias=True)
)

In [5]:
def infer_image(image, model, device, num2char):
    # Preprocess the image
    image = transform_ops(image).unsqueeze(0)  # Add batch dimension

    # Perform inference
    model.eval()
    with torch.no_grad():
        text_logits = model(image)  # [T, batch_size, num_chars]
        text_logits = text_logits.cpu()

    # Decode predictions
    text_pred = decode_predictions(text_logits, num2char)[0]

    # Apply post-processing corrections
    text_pred_corrected = correct_prediction(text_pred)
    
    return text_pred_corrected.replace("!","")
    # return text_pred_corrected

In [6]:
# # Define the transformation operations
# transform_ops = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
# ])

# # Function to remove duplicates in the predicted text
# def remove_duplicates(text):
#     if len(text) > 1:
#         letters = [text[0]] + [letter for idx, letter in enumerate(text[1:], start=1) if text[idx] != text[idx-1]]
#     elif len(text) == 1:
#         letters = [text[0]]
#     else:
#         return ""
#     return "".join(letters)

# # Function to correct the predicted word
# def correct_prediction(word):
#     parts = word.split("-")
#     parts = [remove_duplicates(part) for part in parts]
#     corrected_word = "".join(parts)
#     return corrected_word

# # Function to decode predictions from logits
# def decode_predictions(text_batch_logits):
#     text_batch_tokens = F.softmax(text_batch_logits, 2).argmax(2)  # [T, batch_size]
#     text_batch_tokens = text_batch_tokens.numpy().T  # [batch_size, T]

#     text_batch_tokens_new = []
#     for text_tokens in text_batch_tokens:
#         text = [num2char[str(idx)] for idx in text_tokens]  # Ensure num2char is defined
#         text = "".join(text)
#         text_batch_tokens_new.append(text)

#     return text_batch_tokens_new

# # Inference function
# def infer_image(image, model, device):
#     # Preprocess the image
#     image = transform_ops(image).unsqueeze(0).to(device)  # Add batch dimension

#     # Perform inference
#     model.eval()
#     with torch.no_grad():
#         text_logits = model(image)  # [T, batch_size, num_chars]
#         text_logits = text_logits.cpu()

#     # Decode predictions
#     text_pred = decode_predictions(text_logits)[0]

#     # Apply post-processing corrections
#     text_pred_corrected = correct_prediction(text_pred)

#     return text_pred_corrected


In [7]:
# # Create the Gradio interface
# iface = gr.Interface(
#     fn=image_to_text, 
#     inputs=gr.Image(type="pil"), 
#     outputs="text",
#     title="Image to Text Converter",
#     description="Upload an image and get the extracted text."
# )

# # Launch the interface
# iface.launch(share=True)

In [8]:

def image_to_text(img):
    # Dummy function to simulate image-to-text conversion
    # Replace this with actual image processing and text extraction logic
    # corrected_word = infer_image(img, model, device)
    # return corrected_word
# Define the Gradio inference function
# def image_to_text(img):
    predicted_text = infer_image(img, model, torch.device('cpu'), num2char)
    return predicted_text
    # num2char = num_chars
    # predicted_text = infer_image(img, model, device, num2char)
    # return predicted_text

# Create the Gradio interface
iface = gr.Interface(
    fn=image_to_text, 
    inputs=gr.Image(type="pil"), 
    outputs="text",
    title="Khmer Optical Character Recognition",
    # description="Upload an image and get the extracted text."
)

# Launch the interface
iface.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
